# Un pipeline, dos encoders · DINO E1–E4
**Estado: implementación prospectiva, sin resultados de esta secuencia.** Objetivo: replicar la secuencia de adaptación usada con CLIP. Pregunta: ¿el MLP y su adaptación posterior mediante LoRA mejoran ambos encoders? Hipótesis: el beneficio puede depender del encoder; publicar también resultados negativos. Controles: DINO congelado sin recorte y con YOLO; E3 MLP; E4 LoRA sobre el MLP ganador congelado. Tres semillas por etapa entrenada. No confundir esta secuencia con la ablación lineal/conjunta anterior.
Métodos externos: [CLIP](https://arxiv.org/abs/2103.00020), [DINOv2](https://arxiv.org/abs/2304.07193), [YOLOv8](https://docs.ultralytics.com/models/yolov8/), [LoRA](https://arxiv.org/abs/2106.09685), [batch-hard triplet](https://arxiv.org/abs/1703.07737). La secuencia, controles y selección son decisiones de esta investigación. No se atribuye novedad a estas técnicas. DINO usa CLS 1024; CLIP embedding 512 y una proyección visual propia, también entrenable en su E4 histórico. Esto replica etapas, no capacidad ni parámetros idénticos.
## Fase 1. Preparación y verificación de datos
Corpus preparado v2: 193.276 train, 273 val; separación por identidad. Test público v1 inmutable: 277 fotos, 60 identidades; PetTrace v6: 502 fotos, 50 identidades, separado. Las cajas v2 conservan los splits v1. Sin limpieza aquí. Instalar previamente el código revisado mediante tools/setup_dino_colab.py; CUDA. No cambiar hiperparámetros según test. El test ya influyó en investigar DINO: estudio retrospectivo, no confirmación independiente.


In [ ]:
import sys, os, json, gc, shutil, time
from pathlib import Path
from dataclasses import asdict
if not (3,11) <= sys.version_info[:2] < (3,14):
    raise RuntimeError('Python 3.11–3.13 required')
import numpy as np
import torch
import mlflow
from pettrace.tracking import PRODUCTION_URI, source_identity, publish_evaluation
from pettrace.data.verification import sha256
if not torch.cuda.is_available():raise RuntimeError('The full study requires CUDA')
for name in ('HF_TOKEN','MLFLOW_TRACKING_USERNAME','MLFLOW_TRACKING_PASSWORD'):
    if not os.environ.get(name):
        from google.colab import userdata
        os.environ[name]=userdata.get(name)
    if not os.environ[name]:raise RuntimeError('Missing '+name)
os.environ['MLFLOW_TRACKING_URI']=PRODUCTION_URI
os.environ['MLFLOW_REGISTRY_URI']=PRODUCTION_URI
mlflow.set_tracking_uri(PRODUCTION_URI);mlflow.set_registry_uri(PRODUCTION_URI)
LOG_TO_MLFLOW=True
REGISTER_MODEL=True
DATASET_COMMIT = "52bfe26384c4230008fe4707aa0cb145988766a8"
SEEDS=(45,46,47)
MODES=('mlp','mlp_lora')
REPORT=Path('reports/dino-sequential/20261119-v1')
REPORT.mkdir(parents=True,exist_ok=True)
SOURCE=source_identity()
torch.set_num_threads(4)
from pettrace.data import training
from pettrace.experiments.e3 import load_benchmarks
from pettrace.evaluation.core import load_model
from pettrace.training.dinov2 import build, Config, Pipeline, Preprocessor, fit, REVISION, REPOSITORY
from pettrace.training import triplet, lora
from pettrace.experiments.dinov2 import frozen_features, publish_training
prepared=training(version='v2',cache_dir=str(REPORT/'data'))
assert prepared.fingerprint=='d0ca5c0b1bed8844f943b9fd355ce912524797d0dc71b50c7a1624e39f49fe58'
rows=[r for r in prepared.rows if r['split'] in ('train','val')]
assert sum(r['split']=='train' for r in rows)==193276
assert sum(r['split']=='val' for r in rows)==273
benchmarks=load_benchmarks(prepared,REPORT,private_version='v6',private_commit=DATASET_COMMIT)
base=load_model('models:/pettrace-embedder/2');assert base.golden_ok
crop=lora.PreparedPreprocessor(base.pipeline,prepared)
gate=[next(r for r in rows if r['split']=='train' and r['species']==sp) for sp in ('cat','dog')]
crop.validate(gate)
study=dict(source=SOURCE,model_revision=REVISION,prepared=prepared.provenance,seeds=SEEDS,modes=MODES,private_commit=DATASET_COMMIT,register_model=REGISTER_MODEL,selection='MLP seed by macro validation mAP; fixed MLP parent for all LoRA seeds; epoch zero E3 eligible in E4; no test selection',training_budget='30 epochs x 100 steps maximum per stage; patience 7; P8K4',mlp_architecture='1024-256-1024 ReLU L2',mlp_lr=1e-3,lora_lr=1e-5,public_test_commit='a724c1099a5432609c853958e88b68c4cf82d871')
# Reject reuse after code/configuration changes; use a new study directory instead.
study=json.loads(json.dumps(study))
if (REPORT/'study.json').exists():assert json.loads((REPORT/'study.json').read_text())==study
else:(REPORT/'study.json').write_text(json.dumps(study,indent=2))
print('Full data verified. Study:',REPORT)


## Fase 2. Representación y entrenamiento
E3 reutiliza exactamente Head y fit de CLIP, parametrizados por dimensión: 1024→256→1024, ReLU, L2. El ancho oculto 256 es fijo en ambos encoders, no igualdad de capacidad. Features congeladas; AdamW lr 1e-3, margen 0,2, P8×K4, máximo 30×100 pasos, paciencia 7. Seleccionar el MLP por mAP macro de validación (desempate por semilla), nunca test. E4 usa ese mismo MLP congelado en todas las semillas, query/value LoRA rango 8, alpha 16, lr 1e-5. Época cero reproduce E3 y es elegible. No se agrega una proyección visual artificial a DINO. El E4 histórico CLIP incluye su proyección nativa: declarar esa diferencia y usar su ablación solo-LoRA para aislarla. Publicación obligatoria en MLflow, con reintento sin entrenar. Cada etapa conserva hashes, curvas, coste y parámetros.


In [ ]:
from pettrace.evaluation.metrics import retrieval
from pettrace.models.golden import golden_images, compare, fingerprint
from huggingface_hub import snapshot_download
model_dir=Path(snapshot_download(REPOSITORY,revision=REVISION,allow_patterns=['model.safetensors','config.json','preprocessor_config.json']))
base_hashes={f:sha256(model_dir/f) for f in ('model.safetensors','config.json','preprocessor_config.json')}
base_hashes.update({'detector.'+k:v for k,v in base.sha256.items() if 'detector' in k})
encoder,processor=build('frozen','cuda')
pipeline=Pipeline(encoder,processor,base.pipeline.detector)
control_golden=pipeline.embed(golden_images());np.save(REPORT/'control-golden.npy',control_golden)
x=frozen_features(pipeline,rows,REPORT/'features',dict(model=base_hashes,processor=processor.to_dict(),source=SOURCE,detector_spec=base.pipeline.spec))
vi=[i for i,r in enumerate(rows) if r['split']=='val'];val=[rows[i] for i in vi]
control_metrics=retrieval(x[vi],[r['animal'] for r in val],[r['species'] for r in val])
control={k:float(np.mean([control_metrics[sp][k] for sp in ('cat','dog')])) for k in ('mAP','rank1')}
(REPORT/'control-validation.json').write_text(json.dumps(control_metrics,indent=2))
del encoder,pipeline;gc.collect();torch.cuda.empty_cache()
candidates=[]
for seed in SEEDS:
    folder=REPORT/f'mlp-{seed}';folder.mkdir(exist_ok=True)
    config=triplet.Config(seed=seed)
    started=time.monotonic()
    if (folder/'training-result.json').exists():
        result=json.loads((folder/'training-result.json').read_text())
    else:
        if (folder/'head.pt').exists():
            raise RuntimeError('Incomplete MLP training: preserve this folder and use a new study ID')
        _,result=triplet.fit(x,rows,folder,config,device='cpu',dimension=1024)
        (folder/'training-result.json').write_text(json.dumps(result,indent=2))
        (folder/'cost.json').write_text(json.dumps(dict(training_seconds=time.monotonic()-started,backend='cpu'),indent=2))
    checkpoint=folder/'head.pt'
    saved=torch.load(checkpoint,weights_only=True,map_location='cpu')
    head=triplet.Head(config.hidden,dimension=1024);head.load_state_dict(saved['state_dict'])
    _,values=triplet.validation_score(head,x[vi],val,'cpu')
    metrics={k:float(np.mean([values[sp][k] for sp in ('cat','dog')])) for k in ('mAP','rank1')}
    design=dict(study=study,mode='mlp',seed=seed,config=asdict(config),checkpoint_sha256=sha256(checkpoint),trainable_parameters=lora.trainable_parameters(head),base_hashes=base_hashes)
    training_run=publish_training(folder,design,metrics)
    candidates.append(dict(mode='mlp',seed=seed,validation=metrics,folder=str(folder),training_run=training_run))
parent=min(candidates,key=lambda c:(-c['validation']['mAP'],c['seed']))
parent_path=Path(parent['folder'])/'head.pt'
parent_hash=sha256(parent_path)
(REPORT/'mlp-selection.json').write_text(json.dumps(dict(candidate=parent,sha256=parent_hash),indent=2))
def parent_head():
    assert sha256(parent_path)==parent_hash
    saved=torch.load(parent_path,weights_only=True,map_location='cpu')
    head=triplet.Head(saved['config']['hidden'],dimension=saved['dimension'])
    head.load_state_dict(saved['state_dict'])
    return head
encoder,processor=build('mlp','cuda',head=parent_head())
e3_golden=Pipeline(encoder,processor,base.pipeline.detector).embed(golden_images())
del encoder;gc.collect();torch.cuda.empty_cache()
for seed in SEEDS:
    folder=REPORT/f'mlp_lora-{seed}';folder.mkdir(exist_ok=True)
    # Frozen parent is part of the complete model, not part of adapter.pt.
    shutil.copy2(parent_path,folder/'head.pt')
    config=Config(seed=seed,learning_rate=1e-5)
    encoder,processor=build('mlp_lora','cuda',config,head=parent_head())
    pipeline=Pipeline(encoder,processor,base.pipeline.detector)
    assert compare(e3_golden,pipeline.embed(golden_images()))[0]
    started=time.monotonic();torch.cuda.reset_peak_memory_stats()
    if (folder/'training-result.json').exists():
        result=json.loads((folder/'training-result.json').read_text());lora.load_adapter(encoder,folder/'adapter.pt')
    else:
        encoder,result=fit(encoder,rows,Preprocessor(processor,base.pipeline.detector,crop),folder,config,'cuda')
        (folder/'cost.json').write_text(json.dumps(dict(training_and_validation_seconds=time.monotonic()-started,peak_cuda_bytes=torch.cuda.max_memory_allocated(),backend='cuda'),indent=2))
    _,values=lora._validation_score(encoder,val,Preprocessor(processor,base.pipeline.detector,crop).pixels(val),'cuda')
    metrics={k:float(np.mean([values[sp][k] for sp in ('cat','dog')])) for k in ('mAP','rank1')}
    design=dict(study=study,mode='mlp_lora',seed=seed,config=asdict(config),checkpoint_sha256=sha256(folder/'adapter.pt'),parent_checkpoint_sha256=parent_hash,parent_training_run=parent['training_run'],trainable_parameters=lora.trainable_parameters(encoder),base_hashes=base_hashes)
    training_run=publish_training(folder,design,metrics)
    candidates.append(dict(mode='mlp_lora',seed=seed,validation=metrics,folder=str(folder),training_run=training_run))
    del encoder,pipeline;gc.collect();torch.cuda.empty_cache()
# Registration choice only; every candidate is evaluated and published regardless.
winner=min(candidates,key=lambda c:(-c['validation']['mAP'],c['mode'],c['seed']))
selection=dict(mode=winner['mode'],seed=winner['seed'],parent=parent,selection_metric='validation macro species mAP')
if control['mAP']>=winner['validation']['mAP']:selection.update(mode='frozen',seed=45)
(REPORT/'selection.json').write_text(json.dumps(selection,indent=2))
(REPORT/'candidates.json').write_text(json.dumps(candidates,indent=2))
print('Selection frozen before test:',selection)


## Fase 3. Evaluación y análisis
Evaluar E1 sin recorte, E2 con YOLO, los tres MLP E3 y las tres adaptaciones E4, por separado en público y PetTrace: 16 publicaciones. Galería cerrada de misma especie, leave-one-out por fotografía. Rank-k por máximo de identidad, mAP por fotografías; EER descriptiva. Reportar media y desviación entre semillas, comparación pareada E2→E3 y E3 seleccionado→E4, rescates y daños e intervalos por identidad. Las semillas E4 comparten un solo padre E3: no estiman variabilidad de la secuencia completa. Los resultados CLIP históricos sobre v5 no son comparables con v6: recalcular con iguales snapshots antes de construir la tabla de dos modelos. Registro opcional separado, sin aliases de serving. Limitaciones: capacidad, preentrenamiento, resoluciones, costes y proyección nativa distintos; evaluación retrospectiva, sin separación garantizada por sesión ni prueba de búsqueda abierta. E5 permanece exploratorio. Resultados: se generan abajo; no hay mejoras nuevas acreditadas hasta publicación.


In [ ]:
from pettrace.models.artifacts import Model
from pettrace.evaluation.core import evaluate
from pettrace.evaluation.figures import save
from pettrace.experiments.zero_shot import save_publication_input,load_publication_input
client=mlflow.MlflowClient();publications=[]
for candidate in [dict(mode=m,seed=45,folder=str(REPORT/m)) for m in ('raw','frozen')]+candidates:
    mode,seed=candidate['mode'],candidate['seed'];folder=Path(candidate['folder']);folder.mkdir(exist_ok=True)
    head=None
    hashes=dict(base_hashes)
    if mode in MODES:
        saved=torch.load(folder/'head.pt',weights_only=True,map_location='cpu')
        head=triplet.Head(saved['config']['hidden'],dimension=saved['dimension'])
        head.load_state_dict(saved['state_dict']);hashes['head.pt']=sha256(folder/'head.pt')
    encoder,processor=build('frozen' if mode=='raw' else mode,'cuda',Config(seed=seed),head=head)
    if mode=='mlp_lora':
        lora.load_adapter(encoder,folder/'adapter.pt');hashes['adapter.pt']=sha256(folder/'adapter.pt')
    pipeline=Pipeline(encoder,processor,None if mode=='raw' else base.pipeline.detector)
    golden=pipeline.embed(golden_images());assert compare(golden,pipeline.embed(golden_images()))[0]
    np.save(folder/'golden.npy',golden)
    model=Model(f'pettrace-dino://{REPORT.name}/{mode}/{seed}',str(folder),pipeline,f'DINO-{mode}-{seed}','',hashes,True,fingerprint(golden))
    for domain,ds in (('public',benchmarks.public),('private',benchmarks.private)):
        target=folder/domain;target.mkdir(exist_ok=True)
        if (target/'evaluation-input.json').exists():
            ev=load_publication_input(target);ev.dataset=ds
        else:
            ev=evaluate(ds,pipeline,zero_shot=True);save(ev,model,str(target));save_publication_input(ev,target)
            (target/'runtime.json').write_text(json.dumps(dict(**SOURCE,execution_id=REPORT.name,backend='torch_cuda',model_sha256=hashes),indent=2))
        run=publish_evaluation(ev,model,target,SOURCE['source_sha256'],experiment='DINO-'+mode,expected_model_uri=model.uri,expected_model_sha256=hashes,expected_dataset_version=ds.version,expected_protocol='zero-shot')
        assert client.get_run(run).info.status=='FINISHED'
        receipt=dict(mode=mode,seed=seed,domain=domain,run_id=run,published=True)
        (target/'publication.json').write_text(json.dumps(receipt,indent=2));client.log_artifact(run,str(target/'publication.json'),'report')
        client.log_artifact(run,str(REPORT/'selection.json'),'study')
        publications.append(receipt)
        if domain=='private' and mode==selection['mode'] and (mode=='frozen' or seed==selection['seed']):selected_run=run;selected_folder=folder
    del pipeline,encoder;gc.collect();torch.cuda.empty_cache()
assert len(publications)==16
(REPORT/'publication.json').write_text(json.dumps(publications,indent=2))
from pettrace.evaluation.paired import paired_comparison
analysis={}
for domain in ('public','private'):
    baseline=json.loads((REPORT/'frozen'/domain/'evaluation-input.json').read_text())['result']
    mlp_baseline=json.loads((Path(parent['folder'])/domain/'evaluation-input.json').read_text())['result']
    raw=json.loads((REPORT/'raw'/domain/'evaluation-input.json').read_text())['result']
    analysis[domain]={'E1_to_E2':paired_comparison(raw,baseline)}
    for mode in MODES:
        evaluations=[json.loads((REPORT/f'{mode}-{seed}'/domain/'evaluation-input.json').read_text())['result'] for seed in SEEDS]
        analysis[domain][mode]=dict(metrics={k:dict(mean=float(np.mean([e['test']['global'][k] for e in evaluations])),sd_seeds=float(np.std([e['test']['global'][k] for e in evaluations],ddof=1))) for k in ('rank1','rank2','rank5','mAP')},paired_vs_frozen={str(seed):paired_comparison(baseline,e) for seed,e in zip(SEEDS,evaluations)})
    analysis[domain]['E3_to_E4']={str(seed):paired_comparison(mlp_baseline,json.loads((REPORT/f'mlp_lora-{seed}'/domain/'evaluation-input.json').read_text())['result']) for seed in SEEDS}
(REPORT/'comparison.json').write_text(json.dumps(analysis,indent=2))
for receipt in publications:client.log_artifact(receipt['run_id'],str(REPORT/'comparison.json'),'study')
if REGISTER_MODEL and not (REPORT/'registration.json').exists():
    bundle=selected_folder/'research-model';bundle.mkdir(exist_ok=True)
    for name in ('adapter.pt','head.pt','golden.npy'):
        if (selected_folder/name).exists():shutil.copy2(selected_folder/name,bundle/name)
    (bundle/'model.json').write_text(json.dumps(dict(selection=selection,repository=REPOSITORY,revision=REVISION,processor=processor.to_dict(),detector_uri='models:/pettrace-embedder/2',base_hashes=base_hashes,serving_compatible=False),indent=2))
    import inspect
    import pettrace.training.dinov2 as implementation
    for module in (implementation,triplet,lora):
        shutil.copy2(inspect.getfile(module),bundle/Path(inspect.getfile(module)).name)
    client.log_artifacts(selected_run,str(bundle),'research-model')
    registered=mlflow.register_model(f'runs:/{selected_run}/research-model','pettrace-dinov2-research')
    (REPORT/'registration.json').write_text(json.dumps(dict(name=registered.name,version=registered.version,alias_promoted=False),indent=2))
print('All evaluations published; registration is separate; no serving alias promoted.')
